<a href="https://colab.research.google.com/github/kigemmanuel/Bliss-Exchange/blob/main/BlenderKit_Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BlenderProc and BlenderKit Setup
This notebook sets up BlenderProc to download models from BlenderKit without needing a local Blender installation.

In [ ]:
!pip install blenderproc

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.5/371.5 kB 6.8 MB/s eta 0:00:00
  Created wheel for progressbar: filename=progressbar-2.5-py3-none-any.whl size=12065 sha256=434ade257ba412405abb17a3d2cd06cee36f1d8e2810adfa259567c12a2fc7bd
  Stored in directory: /root/.cache/pip/wheels/a5/4d/c7/f3cf0f75c746c219090060131fe00f1523cc2c5484991f4030
Successfully built progressbar


### Download a BlenderKit Model
You will need an API key from [BlenderKit](https://www.blenderkit.com/become-creator/api-key/).

The following code uses BlenderProc's `loader.load_blenderkit` module to fetch a model by its asset ID or search parameters.

In [ ]:
%%writefile download_model.py
import os
import json
import uuid
import urllib.request
from urllib.request import urlretrieve, build_opener, install_opener
import argparse

def download_specific_asset(query_string, api_key, output_filename):
    opener = build_opener()
    opener.addheaders = [
        ('Accept', 'application/json'),
        ('Authorization', f'Bearer {api_key}'),
        ('User-Agent', 'BlenderKit-Client/1.0')
    ]
    install_opener(opener)

    # Encode the query string for URL safety
    encoded_query = urllib.parse.quote(query_string)
    search_url = f"https://www.blenderkit.com/api/v1/search/?query={encoded_query}"

    print(f"Searching with query: {query_string}")
    try:
        with urllib.request.urlopen(search_url) as response:
            data = json.loads(response.read().decode())
            results = data.get('results', [])
            if not results:
                raise Exception(f"No results found for query: {query_string}")
            asset = results[0]
    except Exception as e:
        raise Exception(f"Search failed: {e}")

    download_url = asset.get('downloadUrl')
    if not download_url:
        # Try to find in files list if not at top level
        for f in asset.get('files', []):
            if f.get('fileType') == 'blend':
                download_url = f.get('downloadUrl')
                break

    if not download_url:
        raise Exception("No download URL found. The asset might be restricted or require a Full Plan.")

    scene_uuid = str(uuid.uuid4())
    print("Requesting download authorization...")
    try:
        with urllib.request.urlopen(f"{download_url}?scene_uuid={scene_uuid}") as response:
            file_data = json.loads(response.read().decode())
            actual_file_path = file_data['filePath']

        print(f"Downloading to {output_filename}...")
        urlretrieve(actual_file_path, output_filename)
        print("Download complete.")
    except Exception as e:
        raise Exception(f"Failed to process download link: {e}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--api_key', required=True)
    args = parser.parse_args()
    # Using the specific search string provided by the user
    query = 'asset_base_id:9e001d9b-c067-4340-a1f1-868b0ff10f08 asset_type:model'
    try:
        download_specific_asset(query, args.api_key, 'downloaded_model.blend')
    except Exception as e:
        print(f"Error: {e}")

Overwriting download_model.py


In [ ]:
# Re-run with debug logging
api_key = "28e0e3cd48fb0d1144e4d294ba7c266f903f0598"
!python3 download_model.py --api_key {api_key}

from google.colab import files
import os
if os.path.exists('downloaded_model.blend'):
    files.download('downloaded_model.blend')
else:
    print('Model not found. Please verify if this asset is "Free" or "Full Plan" only.')

Fetching metadata for asset: 9e001d9b-c067-4340-a1f1-868b0ff10f08
Direct fetch failed, trying search... HTTP Error 404: Not Found
Error: Asset not found in search or detail view.
Model file not found. If this failed, the API key might need to be verified in BlenderKit settings.


In [ ]:
import os
os.environ['DISPLAY'] = '/opt/google/chrome-remote-desktop/start-host --code="4/0AeoWuM8B_N3SdV3zu8DFmLpJw2uda6jcWu2xRSeB5tQG5pWPlAUEUOg97f2rVHavteJ7Yw" --redirect-url="https://remotedesktop.google.com/_/oauthredirect" --name=$(hostname)'
print(f"DISPLAY set to: {os.environ['DISPLAY']}")

In [ ]:
# Re-run the updated download script
api_key = "28e0e3cd48fb0d1144e4d294ba7c266f903f0598"
!python3 download_model.py --api_key {api_key}

from google.colab import files
import os
if os.path.exists('downloaded_model.blend'):
    print("Model found! Downloading to your local machine...")
    files.download('downloaded_model.blend')
else:
    print('Model still not found. This asset ID might be invalid, private, or requires a Full Plan subscription.')

Fetching metadata for asset: 9e001d9b-c067-4340-a1f1-868b0ff10f08
Direct fetch failed, trying search... HTTP Error 404: Not Found
Error: Asset not found in search or detail view.
Model still not found. Please verify the asset status on BlenderKit.


In [ ]:
# Run the final optimized download script with the new query format
api_key = "28e0e3cd48fb0d1144e4d294ba7c266f903f0598"
!python3 download_model.py --api_key {api_key}

from google.colab import files
import os

if os.path.exists('downloaded_model.blend'):
    print("\nSuccess! The model was retrieved. Initiating download to your computer...")
    files.download('downloaded_model.blend')
else:
    print("\nFailed to retrieve the model using the updated query.")
    print("If this still fails, the asset may be restricted to the 'Full Plan' or private.")

Searching with query: asset_base_id:9e001d9b-c067-4340-a1f1-868b0ff10f08 asset_type:model
Requesting download authorization...
Download complete.

Success! The model was retrieved. Initiating download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from google.colab import files

doc_content = """# BlenderKit to Colab: The Complete Troubleshooting & Download Guide

## Table of Contents
1. [Introduction](#1-introduction)
2. [Fresh System Setup & Config](#2-fresh-system-setup--config)
3. [Issues Encountered & Solutions](#3-issues-encountered--solutions)
4. [The Final Working Code](#4-the-final-working-code)
5. [Converting to GLTF/OBJ (with Textures)](#5-converting-to-gltfobj-with-textures)
6. [Verification & Cleanup](#6-verification--cleanup)

## 1. Introduction
This document captures the intensive troubleshooting session required to successfully download assets from BlenderKit via Google Colab. Standard UUID lookups often return 404 errors; this guide provides the specific 'Search Query' workaround that fixed the issue.

## 2. Fresh System Setup & Config
Run these commands in order to prepare a new Colab environment.

### Step A: Install BlenderProc
BlenderProc is preferred because it includes a bundled, headless Blender installation.
```bash
pip install blenderproc
```

### Step B: Headless Display Configuration
To run Blender commands in Colab, you must set a virtual display environment variable. Replace the code below with your specific session token if required, but the variable export is mandatory for `blenderproc run` stability.
```python
import os
os.environ['DISPLAY'] = ':0'
# Note: In our session, we used a Chrome Remote Desktop redirect to initialize the host environment.
```

## 3. Issues Encountered & Solutions

| Problem | Why it happened | The Fix |
| :--- | :--- | :--- |
| **HTTP 404 Error** | Querying the `/assets/ID/` endpoint directly is often restricted. | **Use Search API:** Query the `/search/` endpoint with filters instead. |
| **'Asset Not Found'** | Searching for just the ID string doesn't always trigger the correct filter. | **Specific Query:** Use `asset_base_id:YOUR_ID asset_type:model`. |
| **ModuleNotFoundError** | Running `bproc.loader` directly inside a notebook often hits environment conflicts. | **Stand-alone Script:** Use a custom Python script with `urllib` to handle the API directly. |

## 4. The Final Working Code

### Step A: Create the Downloader (`download_model.py`)
```python
import os, json, uuid, urllib.request, argparse
from urllib.request import urlretrieve, build_opener, install_opener

def download_asset(query_string, api_key, output_filename):
    opener = build_opener()
    opener.addheaders = [('Authorization', f'Bearer {api_key}'), ('User-Agent', 'BlenderKit-Client/1.0')]
    install_opener(opener)

    encoded_query = urllib.parse.quote(query_string)
    url = f"https://www.blenderkit.com/api/v1/search/?query={encoded_query}"

    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read().decode())
        if not data['results']:
            raise Exception("Asset not found.")
        asset = data['results'][0]

    dl_url = asset.get('downloadUrl')
    scene_uuid = str(uuid.uuid4())
    with urllib.request.urlopen(f\"{dl_url}?scene_uuid={scene_uuid}\") as r:
        file_path = json.loads(r.read().decode())['filePath']

    urlretrieve(file_path, output_filename)

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--api_key', required=True)
    args = parser.parse_args()
    # CRITICAL: Search query that bypassed the 404 error
    query = 'asset_base_id:9e001d9b-c067-4340-a1f1-868b0ff10f08 asset_type:model'
    download_asset(query, args.api_key, 'downloaded_model.blend')
```

## 5. Converting to GLTF/OBJ
```bash
# Export to GLB (Best for textures)
!blenderproc run -e "import bpy; bpy.ops.wm.open_mainfile(filepath='downloaded_model.blend'); bpy.ops.export_scene.gltf(filepath='model.glb', export_format='GLB')"
```

## 6. Verification
1. Run `ls -lh` to ensure `model.glb` is generated.
2. Use `files.download('model.glb')` to retrieve your asset.
"""

with open('BlenderKit_Full_Experience_Guide.md', 'w') as f:
    f.write(doc_content)

print("Full Experience Documentation with System Config generated. Downloading...")
files.download('BlenderKit_Full_Experience_Guide.md')


Full Experience Documentation generated. Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Test Model Conversion
We will now use `blenderproc run` to convert the downloaded `.blend` file into more portable formats like `.glb` (GLTF Binary) and `.obj`. GLB is particularly useful as it packs textures into a single file.

In [ ]:
%%writefile convert_model.py
import blenderproc as bproc
import bpy
import os

def convert():
    input_path = 'downloaded_model.blend'
    output_glb = 'model.glb'
    output_obj = 'model.obj'

    if not os.path.exists(input_path):
        print(f'Error: {input_path} not found.')
        return

    # Load the model
    bpy.ops.wm.open_mainfile(filepath=input_path)

    # Export to GLB (This worked in 4.2)
    print(f'Exporting to {output_glb}...')
    bpy.ops.export_scene.gltf(filepath=output_glb, export_format='GLB')

    # Export to OBJ (Updated for Blender 4.2+)
    print(f'Exporting to {output_obj}...')
    try:
        # In Blender 4.2+, the operator name changed
        bpy.ops.wm.obj_export(filepath=output_obj)
    except AttributeError:
        # Fallback for older versions just in case
        bpy.ops.export_scene.obj(filepath=output_obj)

if __name__ == '__main__':
    convert()

Overwriting convert_model.py


In [ ]:
import os

# Fix for the Matplotlib backend error in headless mode
os.environ['MPLBACKEND'] = 'Agg'

# Execute the conversion script using blenderproc
!blenderproc run convert_model.py

# Verify outputs
print('\n--- Verification ---')
for f in ['model.glb', 'model.obj']:
    if os.path.exists(f):
        print(f'[SUCCESS] Created {f} ({os.path.getsize(f)/1024/1024:.2f} MB)')
    else:
        print(f'[FAILED] {f} was not generated.')

Using blender in /root/blender/blender-4.2.1-linux-x64
Using temporary directory: /dev/shm/blender_proc_7b9837cdd3544896bde5add25852a609
Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)
Read blend: "/content/downloaded_model.blend"
Exporting to model.glb...
INFO Draco mesh compression is available, use library at /root/blender/blender-4.2.1-linux-x64/4.2/python/lib/python3.11/site-packages/libextern_draco.so
09:23:19 | INFO: Starting glTF 2.0 export
09:23:19 | INFO: Extracting primitive: Cylinder.003
09:23:19 | INFO: Primitives created: 1
09:23:19 | INFO: Extracting primitive: Cylinder.006
09:23:19 | INFO: Primitives created: 1
09:23:19 | INFO: Extracting primitive: Cylinder.007
09:23:19 | INFO: Primitives created: 1
09:23:19 | INFO: Extracting primitive: Cylinder.008
09:23:19 | INFO: Primitives created: 1
09:23:19 | INFO: Extracting primitive: Cylinder
09:23:19 | INFO: Primitives created: 1
09:23:19 | INFO: Extracting primitive: Cylinder.002
09:23:19 | INFO: Primitives 

# Blender 4.2+ Conversion Guide (Headless)

To convert `.blend` files in a headless environment like Colab using BlenderProc/Blender 4.2, follow these critical steps:

### 1. Script Structure
BlenderProc requires `import blenderproc` as the **first line**. Without this, the headless environment won't initialize correctly.

### 2. Exporting to GLB (Best for textures)
```python
bpy.ops.export_scene.gltf(filepath='model.glb', export_format='GLB')
```

### 3. Exporting to OBJ (New API for 4.2+)
Blender 4.2 replaced `bpy.ops.export_scene.obj` with a new operator:
```python
bpy.ops.wm.obj_export(filepath='model.obj')
```

### 4. Headless Display
Always set the Matplotlib backend to `Agg` before running `blenderproc run` to avoid display errors:
```python
import os
os.environ['MPLBACKEND'] = 'Agg'
```

In [ ]:
from google.colab import files

converter_docs = """# BlenderKit to GLB/OBJ Converter Guide

## Execution
Run the following command in your terminal/cell:
`blenderproc run convert_model.py`

## convert_model.py content:
import blenderproc as bproc
import bpy
import os

def convert():
    # Load blend
    bpy.ops.wm.open_mainfile(filepath='downloaded_model.blend')

    # Export GLB
    bpy.ops.export_scene.gltf(filepath='model.glb', export_format='GLB')

    # Export OBJ (Blender 4.2+)
    try:
        bpy.ops.wm.obj_export(filepath='model.obj')
    except AttributeError:
        bpy.ops.export_scene.obj(filepath='model.obj')

if __name__ == '__main__':
    convert()
"""

with open('Conversion_Code_Guide.md', 'w') as f:
    f.write(converter_docs)

files.download('Conversion_Code_Guide.md')
print("Documentation generated and download triggered.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Documentation generated and download triggered.


### Unified BlenderKit Pipeline
This script performs the following in one go:
1. Searches for the asset using the `asset_base_id` query.
2. Downloads the `.blend` file named as `{asset_id}.blend`.
3. Uses `blenderproc` to convert that specific file to `{asset_id}.glb` and `{asset_id}.obj`.

In [ ]:
%%writefile unified_pipeline.py
import blenderproc as bproc
import bpy
import os
import json
import uuid
import urllib.request
import argparse
import zipfile
from urllib.request import urlretrieve, build_opener, install_opener

def run_pipeline(api_key, asset_id):
    # 1. DOWNLOAD PHASE
    query_string = f'asset_base_id:{asset_id} asset_type:model'
    output_blend = f'{asset_id}.blend'

    opener = build_opener()
    opener.addheaders = [
        ('Accept', 'application/json'),
        ('Authorization', f'Bearer {api_key}'),
        ('User-Agent', 'BlenderKit-Client/1.0')
    ]
    install_opener(opener)

    encoded_query = urllib.parse.quote(query_string)
    search_url = f"https://www.blenderkit.com/api/v1/search/?query={encoded_query}"

    print(f"Searching for asset: {asset_id}")
    with urllib.request.urlopen(search_url) as response:
        data = json.loads(response.read().decode())
        results = data.get('results', [])
        if not results:
            print(f"Error: No results found for ID {asset_id}")
            return
        asset = results[0]

    download_url = asset.get('downloadUrl')
    if not download_url:
        for f in asset.get('files', []):
            if f.get('fileType') == 'blend':
                download_url = f.get('downloadUrl')
                break

    if not download_url:
        print("Error: No download URL found.")
        return

    scene_uuid = str(uuid.uuid4())
    print("Requesting download authorization...")
    with urllib.request.urlopen(f"{download_url}?scene_uuid={scene_uuid}") as response:
        file_data = json.loads(response.read().decode())
        actual_file_path = file_data['filePath']

    print(f"Downloading to {output_blend}...")
    urlretrieve(actual_file_path, output_blend)

    # 2. CONVERSION PHASE
    if os.path.exists(output_blend):
        print(f"Loading {output_blend} for conversion...")
        bpy.ops.wm.open_mainfile(filepath=output_blend)

        glb_path = f'{asset_id}.glb'
        print(f"Exporting {glb_path}...")
        bpy.ops.export_scene.gltf(filepath=glb_path, export_format='GLB')

        obj_path = f'{asset_id}.obj'
        print(f"Exporting {obj_path}...")
        try:
            bpy.ops.wm.obj_export(filepath=obj_path)
        except AttributeError:
            bpy.ops.export_scene.obj(filepath=obj_path)

        # 3. ZIP PHASE
        zip_filename = f"{asset_id}.zip"
        print(f"Creating archive: {zip_filename}...")
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for ext in ['blend', 'glb', 'obj']:
                fname = f"{asset_id}.{ext}"
                if os.path.exists(fname):
                    zipf.write(fname)

        print("Pipeline processing complete.")
    else:
        print("Download failed, skipping conversion.")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--api_key', required=True)
    parser.add_argument('--asset_id', required=True)
    args = parser.parse_args()
    run_pipeline(args.api_key, args.asset_id)

Overwriting unified_pipeline.py


In [ ]:
import os
from google.colab import files

# CONFIGURATION
API_KEY = "28e0e3cd48fb0d1144e4d294ba7c266f903f0598"
ASSET_ID = "9e001d9b-c067-4340-a1f1-868b0ff10f08"

# Run the updated unified script with Zipping support
os.environ['MPLBACKEND'] = 'Agg'
!blenderproc run unified_pipeline.py --api_key {API_KEY} --asset_id {ASSET_ID}

# Verification and download of the ZIP
zip_filename = f"{ASSET_ID}.zip"
if os.path.exists(zip_filename):
    print(f"\nSuccess! Downloading archive: {zip_filename} ({os.path.getsize(zip_filename)/1024/1024:.2f} MB)...")
    files.download(zip_filename)
else:
    print(f"\nError: {zip_filename} was not generated.")

Using blender in /root/blender/blender-4.2.1-linux-x64
Using temporary directory: /dev/shm/blender_proc_90ac4b15b96647baa91291a07c095151
Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)
Searching for asset: 9e001d9b-c067-4340-a1f1-868b0ff10f08
Requesting download authorization...
Loading 9e001d9b-c067-4340-a1f1-868b0ff10f08.blend for conversion...
Read blend: "/content/9e001d9b-c067-4340-a1f1-868b0ff10f08.blend"
Exporting 9e001d9b-c067-4340-a1f1-868b0ff10f08.glb...
INFO Draco mesh compression is available, use library at /root/blender/blender-4.2.1-linux-x64/4.2/python/lib/python3.11/site-packages/libextern_draco.so
09:48:10 | INFO: Starting glTF 2.0 export
09:48:10 | INFO: Extracting primitive: Cylinder.003
09:48:10 | INFO: Primitives created: 1
09:48:10 | INFO: Extracting primitive: Cylinder.006
09:48:10 | INFO: Primitives created: 1
09:48:10 | INFO: Extracting primitive: Cylinder.007
09:48:10 | INFO: Primitives created: 1
09:48:10 | INFO: Extracting primitive: Cylinde

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### External System Requirements
To run this script on your own machine (outside of Colab), you will need:
1. **Python 3.10+**
2. **BlenderProc**: Install via `pip install blenderproc`.
3. **Blender**: BlenderProc will usually download a headless version automatically on first run, but having Blender 4.2+ installed is recommended.
4. **Internet Connection**: To access the BlenderKit API.

In [ ]:
%%writefile unified_multi_pipeline.py
import blenderproc as bproc
import bpy
import os
import json
import uuid
import urllib.request
import argparse
import zipfile
import shutil
from urllib.request import urlretrieve, build_opener, install_opener

def run_pipeline(api_key, asset_id):
    workspace = os.path.join(os.getcwd(), f"work_{asset_id}")
    os.makedirs(workspace, exist_ok=True)

    output_blend = os.path.join(workspace, f'{asset_id}.blend')
    glb_path = os.path.join(workspace, f'{asset_id}.glb')
    obj_path = os.path.join(workspace, f'{asset_id}.obj')
    zip_filename = f"{asset_id}.zip"

    query_string = f'asset_base_id:{asset_id} asset_type:model'
    opener = build_opener()
    opener.addheaders = [('Accept', 'application/json'), ('Authorization', f'Bearer {api_key}'), ('User-Agent', 'BlenderKit-Client/1.0')]
    install_opener(opener)

    encoded_query = urllib.parse.quote(query_string)
    search_url = f"https://www.blenderkit.com/api/v1/search/?query={encoded_query}"

    print(f"\n--- Processing Asset: {asset_id} ---")
    try:
        with urllib.request.urlopen(search_url) as response:
            data = json.loads(response.read().decode())
            asset = data.get('results', [])[0]
            download_url = asset.get('downloadUrl')
            if not download_url:
                for f in asset.get('files', []):
                    if f.get('fileType') == 'blend':
                        download_url = f.get('downloadUrl')
                        break

            scene_uuid = str(uuid.uuid4())
            with urllib.request.urlopen(f"{download_url}?scene_uuid={scene_uuid}") as dl_res:
                actual_file_path = json.loads(dl_res.read().decode())['filePath']
                urlretrieve(actual_file_path, output_blend)
    except Exception as e:
        print(f"Failed download for {asset_id}: {e}")
        return

    if os.path.exists(output_blend):
        bpy.ops.wm.open_mainfile(filepath=output_blend)
        bpy.ops.export_scene.gltf(filepath=glb_path, export_format='GLB')
        try:
            bpy.ops.wm.obj_export(filepath=obj_path)
        except:
            bpy.ops.export_scene.obj(filepath=obj_path)

        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for fpath in [output_blend, glb_path, obj_path]:
                if os.path.exists(fpath):
                    zipf.write(fpath, arcname=os.path.basename(fpath))

        shutil.rmtree(workspace)
        print(f"Cleanup complete. ZIP created: {zip_filename}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--api_key', required=True)
    parser.add_argument('--asset_ids', required=True)
    args = parser.parse_args()
    ids = [i.strip() for i in args.asset_ids.split(',')]
    for aid in ids:
        run_pipeline(args.api_key, aid)

Writing unified_multi_pipeline.py


In [ ]:
import os
from google.colab import files

# CONFIGURATION - Add as many IDs as you want to the array
API_KEY = "28e0e3cd48fb0d1144e4d294ba7c266f903f0598"
ASSET_ID_ARRAY = ["9e001d9b-c067-4340-a1f1-868b0ff10f08"]

ids_str = ",".join(ASSET_ID_ARRAY)
os.environ['MPLBACKEND'] = 'Agg'
!blenderproc run unified_multi_pipeline.py --api_key {API_KEY} --asset_ids "{ids_str}"

# Download all generated ZIPs
for aid in ASSET_ID_ARRAY:
    zip_file = f"{aid}.zip"
    if os.path.exists(zip_file):
        files.download(zip_file)

# Also download the script file for your records
files.download('unified_multi_pipeline.py')

In [68]:
import os

files_to_keep = [
    'unified_multi_pipeline.py',
    'unified_pipeline.py',
    'download_model.py',
    'BlenderKit_Full_Experience_Guide.md',
    'Conversion_Code_Guide.md',
    'Local_Execution_Guide.md',
    'BlenderKit_Colab_Guide.md' # Assuming this was an important guide
]

# Get all files in the current directory
all_files = os.listdir('./')

print("Starting file cleanup...")

deleted_files = []
kept_files = []

for file_name in all_files:
    # Skip directories and special files like .config
    if os.path.isdir(file_name) or file_name.startswith('.'):
        continue

    if file_name in files_to_keep:
        kept_files.append(file_name)
    elif file_name.endswith(('.zip', '.blend', '.glb', '.obj', '.mtl')) or \
         file_name.startswith('debug_') or \
         file_name == 'convert_model.py' or \
         file_name == 'list_loaders.py' or \
         file_name == 'model.glb' or \
         file_name == 'model.obj' or \
         file_name == 'model.mtl' or \
         file_name == 'downloaded_model.blend':
        try:
            os.remove(file_name)
            deleted_files.append(file_name)
        except OSError as e:
            print(f"Error deleting {file_name}: {e}")
    else:
        kept_files.append(file_name) # Keep other files not explicitly targeted for deletion

print("\nCleanup complete.")
print("\nFiles deleted:")
if deleted_files:
    for f in deleted_files:
        print(f"- {f}")
else:
    print("No files were deleted.")

print("\nFiles remaining (including requested keeps and others not targeted):")
for f in kept_files:
    print(f"- {f}")


Starting file cleanup...

Cleanup complete.

Files deleted:
- 5cd83dec-a4c2-4688-b162-6d173e52cbd4.zip
- debug_script_content.py
- b9a2470f-bfd9-4180-a07f-7c771efd646c.zip
- downloaded_model.blend
- debug_bproc.py
- convert_model.py
- 082d2e0c-c310-4843-9741-6b805d1c6e39.zip
- 0516967a-8de0-42ee-a6c4-86caf08174eb.zip
- 9e001d9b-c067-4340-a1f1-868b0ff10f08.obj
- 712293ab-7a7f-4034-b48e-8ea4d170cd68.zip
- ba718835-c394-4b9f-9256-8c5c40e04165.zip
- 9e001d9b-c067-4340-a1f1-868b0ff10f08.mtl
- model.obj
- 4b5d1cc0-b2ff-4ccd-9244-7427ca58827b.zip
- 2133f91b-d030-4651-9e2b-dd5161249b49.zip
- 799d3086-2eee-4d05-9ed3-3ccfc8060a58.zip
- model.mtl
- a0ee341d-3732-4610-a928-b9658e584de9.zip
- 92cbe435-2cdb-4f0a-84a7-0e6521f150cc.zip
- 0ebc5cfb-b097-4879-a3bd-4ed8aebc1eab.zip
- 74cdfa6a-923f-4f22-bf9f-2467bfa0f0bc.zip
- model.glb
- 020ebba2-cc67-469a-bdf7-a399651277f3.zip
- list_loaders.py
- 9e001d9b-c067-4340-a1f1-868b0ff10f08.glb
- 9e001d9b-c067-4340-a1f1-868b0ff10f08.zip
- b388ed23-9efb-4bcc-9b76

In [ ]:
from google.colab import files
import os

# Ensure scripts exist, then download
scripts = ['unified_multi_pipeline.py', 'download_model.py']
for script in scripts:
    if os.path.exists(script):
        files.download(script)
    else:
        print(f'Error: {script} missing. Running creation...')

File unified_multi_pipeline.py not found in the current workspace.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from google.colab import files
import re

# Configuration
API_KEY = "28e0e3cd48fb0d1144e4d294ba7c266f903f0598"
raw_input = "asset_base_id:c2135d17-36fc-43c2-879e-f1b3d87b1943 asset_type:model, asset_base_id:0516967a-8de0-42ee-a6c4-86caf08174eb asset_type:model, asset_base_id:0ba89eae-2d69-450f-8ce8-d0087cf6ec12 asset_type:model, asset_base_id:ef771313-03bd-4480-8fc3-14301107a24c asset_type:model, asset_base_id:b388ed23-9efb-4bcc-9b76-268ff9fe1768 asset_type:model, asset_base_id:799d3086-2eee-4d05-9ed3-3ccfc8060a58 asset_type:model, asset_base_id:92cbe435-2cdb-4f0a-84a7-0e6521f150cc asset_type:model, asset_base_id:020ebba2-cc67-469a-bdf7-a399651277f3 asset_type:model, asset_base_id:74cdfa6a-923f-4f22-bf9f-2467bfa0f0bc asset_type:model, asset_base_id:082d2e0c-c310-4843-9741-6b805d1c6e39 asset_type:model, asset_base_id:712293ab-7a7f-4034-b48e-8ea4d170cd68 asset_type:model, asset_base_id:2133f91b-d030-4651-9e2b-dd5161249b49 asset_type:model, asset_base_id:ba718835-c394-4b9f-9256-8c5c40e04165 asset_type:model, asset_base_id:4b5d1cc0-b2ff-4ccd-9244-7427ca58827b asset_type:model, asset_base_id:b9a2470f-bfd9-4180-a07f-7c771efd646c asset_type:model, asset_base_id:5cd83dec-a4c2-4688-b162-6d173e52cbd4 asset_type:model, asset_base_id:978b9567-5fb5-493b-9cb9-2b86fa6db650 asset_type:material, asset_base_id:a0ee341d-3732-4610-a928-b9658e584de9 asset_type:model, asset_base_id:c367d0b4-c232-4d67-b968-7505f712100d asset_type:model, asset_base_id:0ebc5cfb-b097-4879-a3bd-4ed8aebc1eab asset_type:model"

# Extract UUIDs using regex
asset_ids = re.findall(r'asset_base_id:([a-f0-9\-]+)', raw_input)
ids_str = ",".join(asset_ids)

print(f"Processing {len(asset_ids)} assets...")

# Run the unified multi pipeline
os.environ['MPLBACKEND'] = 'Agg'
!blenderproc run unified_multi_pipeline.py --api_key {API_KEY} --asset_ids "{ids_str}"

# Download all generated ZIPs
for aid in asset_ids:
    zip_file = f"{aid}.zip"
    if os.path.exists(zip_file):
        files.download(zip_file)
    else:
        print(f"Warning: {zip_file} not found. Asset may be private or restricted.")

Processing 20 assets...
Using blender in /root/blender/blender-4.2.1-linux-x64
Using temporary directory: /dev/shm/blender_proc_da37263efaa044ab953a1ed4b75a6356
Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)

--- Processing Asset: c2135d17-36fc-43c2-879e-f1b3d87b1943 ---
Read blend: "/content/work_c2135d17-36fc-43c2-879e-f1b3d87b1943/c2135d17-36fc-43c2-879e-f1b3d87b1943.blend"
INFO Draco mesh compression is available, use library at /root/blender/blender-4.2.1-linux-x64/4.2/python/lib/python3.11/site-packages/libextern_draco.so
11:33:28 | INFO: Starting glTF 2.0 export
11:33:28 | INFO: Extracting primitive: Cylinder.004
11:33:28 | INFO: Primitives created: 1
11:33:28 | INFO: Extracting primitive: NurbsPath.007
11:33:28 | INFO: Primitives created: 1
11:33:28 | INFO: Extracting primitive: Cylinder.010
11:33:28 | INFO: Primitives created: 3
11:33:28 | INFO: Finished glTF 2.0 export in 0.0383145809173584 s

Writing to /content/work_c2135d17-36fc-43c2-879e-f1b3d87b1943/c213

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import os

if os.path.exists('unified_multi_pipeline.py'):
    files.download('unified_multi_pipeline.py')
else:
    print('Error: unified_multi_pipeline.py not found. Please ensure the creation cell was run.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Instruction Guide for Local Execution

To run the `unified_multi_pipeline.py` script on your local machine, follow these steps:

### 1. Prerequisites
Ensure you have **Python 3.10** or higher installed.

### 2. Install Dependencies
Open your terminal and run:
```bash
pip install blenderproc
```

### 3. Execution Command
You can process one or multiple assets by providing a comma-separated list of IDs:
```bash
blenderproc run unified_multi_pipeline.py --api_key "YOUR_API_KEY" --asset_ids "ID_1,ID_2,ID_3"
```

### 4. How it works
- **Isolation**: The script creates a `work_[ID]` folder for each model to prevent file mixing.
- **Cleanup**: It automatically deletes the raw `.blend`, `.obj`, and `.glb` files after they are safely packed into the `[ID].zip` archive.
- **Results**: You will find the final ZIP files in the same directory where you ran the script.

In [ ]:
from google.colab import files

local_instructions = """# Local Execution Guide: BlenderKit Multi-Pipeline

## Setup
1. Install Python 3.10+
2. Install BlenderProc: `pip install blenderproc`

## Running the Script
Use the following command format in your terminal:

`blenderproc run unified_multi_pipeline.py --api_key "YOUR_API_KEY" --asset_ids "ID1,ID2"`

## Script Features
- **Workspace Isolation**: Creates unique folders per asset during processing.
- **Auto-Cleanup**: Deletes intermediate files (.blend, .obj, .glb) after Zipping.
- **Compatibility**: Uses Blender 4.2+ compatible export operators.
"""

with open('Local_Execution_Guide.md', 'w') as f:
    f.write(local_instructions)

files.download('Local_Execution_Guide.md')
print("Instruction guide generated and download triggered.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Instruction guide generated and download triggered.


In [ ]:
import os

# Fix for the Matplotlib backend error in headless mode
os.environ['MPLBACKEND'] = 'Agg'

# Execute the conversion script using blenderproc
!blenderproc run convert_model.py

# Verify outputs
print('\n--- Verification ---')
for f in ['model.glb', 'model.obj']:
    if os.path.exists(f):
        print(f'[SUCCESS] Created {f} ({os.path.getsize(f)/1024/1024:.2f} MB)')
    else:
        print(f'[FAILED] {f} was not generated.')

Using blender in /root/blender/blender-4.2.1-linux-x64
Using temporary directory: /dev/shm/blender_proc_481ebdbd3b934fef86f76ca6de34bae9
Blender 4.2.1 LTS (hash 396f546c9d82 built 2024-08-19 23:32:23)
Read blend: "/content/downloaded_model.blend"
Exporting to model.glb...
INFO Draco mesh compression is available, use library at /root/blender/blender-4.2.1-linux-x64/4.2/python/lib/python3.11/site-packages/libextern_draco.so
09:22:48 | INFO: Starting glTF 2.0 export
09:22:48 | INFO: Extracting primitive: Cylinder.003
09:22:48 | INFO: Primitives created: 1
09:22:48 | INFO: Extracting primitive: Cylinder.006
09:22:48 | INFO: Primitives created: 1
09:22:48 | INFO: Extracting primitive: Cylinder.007
09:22:48 | INFO: Primitives created: 1
09:22:48 | INFO: Extracting primitive: Cylinder.008
09:22:48 | INFO: Primitives created: 1
09:22:48 | INFO: Extracting primitive: Cylinder
09:22:49 | INFO: Primitives created: 1
09:22:49 | INFO: Extracting primitive: Cylinder.002
09:22:49 | INFO: Primitives 

In [ ]:
from google.colab import files
import os

# Check which files exist and offer them for download
for f in ['model.glb', 'model.obj', 'model.mtl']:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)
    else:
        print(f"File {f} not found, skipping download.")

File model.glb not found, skipping download.
File model.obj not found, skipping download.
File model.mtl not found, skipping download.


In [ ]:
import os

# Execute the conversion script using blenderproc
!blenderproc run convert_model.py

# Verify outputs
print('\n--- Verification ---')
for f in ['model.glb', 'model.obj']:
    if os.path.exists(f):
        print(f'[SUCCESS] Created {f} ({os.path.getsize(f)/1024/1024:.2f} MB)')
    else:
        print(f'[FAILED] {f} was not generated.')

Using blender in /root/blender/blender-4.2.1-linux-x64
Traceback (most recent call last):
  File "/usr/local/bin/blenderproc", line 8, in <module>
    sys.exit(cli())
             ^^^^^
  File "/usr/local/lib/python3.12/dist-packages/blenderproc/command_line.py", line 147, in cli
    SetupUtility.check_if_setup_utilities_are_at_the_top(path_src_run)
  File "/usr/local/lib/python3.12/dist-packages/blenderproc/python/utility/SetupUtility.py", line 405, in check_if_setup_utilities_are_at_the_top
    raise RuntimeError(f'The given script "{path_to_run_file}" does not have a blenderproc '
RuntimeError: The given script "convert_model.py" does not have a blenderproc import at the top! Make sure that is the first thing you import, as otherwise the import of third-party packages installed in the blender environment will fail.
Your code:
#####################
import bpy
import os
""####################
Replaces this with:
""import blenderproc as bproc"

--- Verification ---
[FAILED] model.glb w